# ECE 257 — YOLO Segmentation Photo Tester

Run the cells from top to bottom. When prompted, upload the project's `best.pt`. The last cell opens an interface where you can drag in a photo and inspect the `floor` and `obstacle` segmentation results.

In [ ]:
!pip -q install ultralytics gradio


In [ ]:
from google.colab import files
from pathlib import Path

print('Please upload best.pt')
uploaded = files.upload()
pt_files = [Path(name) for name in uploaded if name.lower().endswith('.pt')]
if not pt_files:
    raise ValueError('No .pt model was uploaded. Please run this cell again and select best.pt.')
MODEL_PATH = str(pt_files[0])
print('Loaded model file:', MODEL_PATH)


In [ ]:
import cv2
import gradio as gr
import numpy as np
from ultralytics import YOLO

model = YOLO(MODEL_PATH)

def analyze(image, confidence):
    if image is None:
        return None, 'Please upload a photo.'
    results = model.predict(image, conf=float(confidence), imgsz=640, verbose=False)
    result = results[0]
    rendered = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
    if result.boxes is None or len(result.boxes) == 0:
        return rendered, 'No floor or obstacle was detected.'
    lines = []
    for index, (class_id, score) in enumerate(zip(result.boxes.cls.tolist(), result.boxes.conf.tolist()), 1):
        label = result.names[int(class_id)]
        lines.append(f'{index}. {label}: confidence {score:.2f}')
    return rendered, '\n'.join(lines)

demo = gr.Interface(
    fn=analyze,
    inputs=[
        gr.Image(type='numpy', label='Upload a test photo'),
        gr.Slider(0.10, 0.90, value=0.25, step=0.05, label='Confidence threshold'),
    ],
    outputs=[
        gr.Image(type='numpy', label='Segmentation result'),
        gr.Textbox(label='Detections'),
    ],
    title='ECE 257 Road and Obstacle Segmentation Tester',
    description='Upload a photo to test the trained floor and obstacle segmentation model.',
)
demo.launch(share=True, debug=False)
